In [1]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from IPython.display import display
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


def rolling_collinearity(X: pd.DataFrame, window: int = 252, step: int = 1):
    """
    係数ローリング回帰と同一窓で VIF・条件数を時系列算出。
    X: 説明変数のみ（被説明変数は含めない）
    """
    cols = X.columns.tolist()
    idx, vif_rows, cond_rows = [], [], []

    for end in range(window, len(X) + 1, step):
        win = X.iloc[end - window : end]
        # 窓内に分散ゼロ列がないか確認（祝日ゼロ埋め等の残存対策）
        if (win.std() == 0).any():
            continue
        Xc = sm.add_constant(win)
        vifs = [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])]
        vif_rows.append(dict(zip(Xc.columns, vifs)))

        Xs = (win - win.mean()) / win.std()
        cond_rows.append(np.linalg.cond(sm.add_constant(Xs).values))
        idx.append(X.index[end - 1])

    vif_ts = pd.DataFrame(vif_rows, index=idx).drop(columns="const")
    cond_ts = pd.Series(cond_rows, index=idx, name="cond_number")
    return vif_ts, cond_ts


def rolling_correlation(X: pd.DataFrame, window: int = 256, step: int = 1):
    """
    説明変数間の全ペア相関を、係数ローリング回帰と同一窓で時系列算出。
    X: 説明変数のみ（被説明変数は含めない）
    戻り値: corr_ts (各列が変数ペア, 値はその窓のpairwise相関)
    """
    from itertools import combinations

    cols = X.columns.tolist()
    pairs = list(combinations(cols, 2))
    idx, rows = [], []

    for end in range(window, len(X) + 1, step):
        win = X.iloc[end - window : end]
        if (win.std() == 0).any():  # 分散ゼロ列（祝日ゼロ埋め残存等）を回避
            continue
        c = win.corr()
        rows.append({f"{a} × {b}": c.loc[a, b] for a, b in pairs})
        idx.append(X.index[end - 1])

    return pd.DataFrame(rows, index=idx)


def run_rolling_factor_regression(
    df_daily_return,
    window=252,
    maxlags=21,
    y_col="Quality",
    x_cols=("Value", "Size", "Momentum", "Low Volatility", "Growth"),
):
    X = sm.add_constant(df_daily_return[list(x_cols)])
    y = df_daily_return[y_col]

    rolling_results = []
    dates = df_daily_return.index

    for i in range(window, len(df_daily_return) + 1):
        X_slice = X.iloc[i - window : i]
        y_slice = y.iloc[i - window : i]
        current_date = dates[i - 1]

        try:
            model = sm.OLS(y_slice, X_slice).fit(
                cov_type="HAC", cov_kwds={"maxlags": maxlags}
            )
            row_data = {"Date": current_date, "R_squared": model.rsquared}
            # 各変数のcoefとz-value(t-value)を格納
            for col in X.columns:
                row_data[f"coef_{col}"] = model.params[col]
                row_data[f"zstat_{col}"] = model.bse[col]

            rolling_results.append(row_data)
        except Exception as e:
            print(f"Error at {current_date}: {e}")
            continue

    df_rolling = pd.DataFrame(rolling_results).set_index("Date")
    return df_rolling


PRJ_DIR = Path().cwd()
parquet_file = PRJ_DIR / "MSCI ACWI_FTW-LS-cum.parquet"
df_cum = pd.read_parquet(parquet_file)

# Value LSとLow Volatiliry LSの符号を反転しい、Low-Highのリターンにする
value_cols = [c for c in df_cum.columns if "Value" in c]
inverted_value_cols = [
    c.replace("Long-Short (High-Low)", "Long-Short (Low-High)") for c in value_cols
]
vol_cols = [c for c in df_cum.columns if "Low Volatility" in c]
inverted_vol_cols = [
    c.replace("Long-Short (High-Low)", "Long-Short (Low-High)") for c in vol_cols
]

df_cum[value_cols] = -1 * df_cum[value_cols]
df_cum.rename(
    columns={key: value for key, value in zip(value_cols, inverted_value_cols)},
    inplace=True,
)

df_cum[vol_cols] = -1 * df_cum[vol_cols]
df_cum.rename(
    columns={key: value for key, value in zip(vol_cols, inverted_vol_cols)},
    inplace=True,
)

# 累積リターンを累積グロスに変換
df_level = df_cum.div(100) + 1.0

df_ret_log = np.log(df_level / df_level.shift(1))
df_ret_log = df_ret_log.dropna(how="all")
display(df_ret_log.head())


/Users/yukihata/Desktop/quants/.venv/lib/python3.12/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/Users/yukihata/Desktop/quants/.venv/lib/python3.12/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


variable,FTW MXWD Index 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Communications Growth Long-Short (High-Low) Total Return,FTW MXWD Index Communications Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Communications Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Communications Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Communications Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Communications Quality Long-Short (High-Low) Total Return,FTW MXWD Index Communications Size Long-Short (High-Low) Total Return,...,FTW MXWD Index Utilities Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Growth Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Low Volatility Long-Short (Low-High) Total Return,FTW MXWD Index Utilities Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Quality Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Size Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Value Long-Short (Low-High) Total Return,FTW MXWD Index Value Long-Short (Low-High) Total Return
Date,,,,,,,,,,,,,,,,,,,,,
2007-01-02,-0.000900,NaN,NaN,NaN,NaN,0.001299,-0.000700,NaN,NaN,NaN,...,NaN,NaN,NaN,0.005385,NaN,NaN,0.007770,0.000300,NaN,0.001199
2007-01-03,-0.003811,0.000300,-0.002503,0.00060,-0.002904,0.004882,-0.000701,NaN,-0.005917,0.006081,...,0.008464,0.002397,0.003295,-0.004086,-0.000100,NaN,-0.010574,-0.006016,-0.016028,-0.000899
2007-01-04,-0.005440,-0.002803,-0.004219,0.01547,-0.002510,0.007426,-0.002707,NaN,-0.003022,0.005649,...,0.002871,-0.011034,-0.016887,-0.006211,-0.003908,NaN,-0.002410,0.004115,-0.016599,-0.002102
2007-01-05,0.001010,0.002203,-0.004440,0.00950,-0.005849,0.001971,-0.006447,NaN,-0.002121,0.001580,...,0.001976,0.003021,-0.003046,-0.021740,-0.025112,NaN,-0.003726,-0.007339,-0.012475,-0.001504
2007-01-06,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000


## ベース多変量回帰

$$ R*{Quality,t} = \alpha + \sum*{factor \in Val, Grwoth, Size, Mom, LowVol} \beta*{factor} R*{factor,t} + \epsilon_t $$


In [2]:
def get_quality_return_df(
    df_log_return: pd.DataFrame,
    target_sector: str,
    factors: list[str] | None = None,
) -> pd.DataFrame:
    if not factors:
        factors = [
            "Momentum",
            "Low Volatility",
            "Size",
            "Value",
            "Growth Long-Short",
            "Quality",
        ]

    target_cols = [
        s
        for s in df_ret_log.columns
        if (f"FTW MXWD Index {target_sector}" in s) and any(f in s for f in factors)
    ]

    ret_quality = df_ret_log[target_cols].copy()

    ret_quality.columns = [
        s.replace(f"FTW MXWD Index {target_sector} ", "")
        .replace(" Long-Short (High-Low) Total Return", "")
        .replace(" Long-Short (Low-High) Total Return", "")
        .replace("Sector Neutralized ", "")
        for s in ret_quality.columns
    ]
    ret_quality = ret_quality[~(ret_quality == 0).all(axis=1).copy()].dropna(how="any")

    return ret_quality


In [3]:
ret_quality = get_quality_return_df(
    df_log_return=df_ret_log, target_sector="Sector Neutralized"
)
display(ret_quality.head())


,Growth,Low Volatility,Momentum,Quality,Size,Value
Date,,,,,,
2007-01-02,-0.000200,0.000900,0.000500,-0.000100,-0.001601,0.002597
2007-01-03,-0.001101,-0.006314,-0.003906,-0.002905,-0.003914,-0.004198
2007-01-04,-0.001102,-0.006962,-0.007352,-0.001707,-0.000201,-0.004719
2007-01-05,0.001903,-0.002737,-0.004356,0.000502,-0.002820,-0.002116
2007-01-08,0.000400,-0.000609,0.000000,0.000803,-0.001514,0.000000


#### 1期間OLS


In [4]:
def run_factor_regression(
    ret_quality,
    y_col="Quality",
    x_cols=("Value", "Size", "Momentum", "Low Volatility"),
):
    X = sm.add_constant(ret_quality[list(x_cols)])
    y = ret_quality[y_col]
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 21})

    return model


model_full = run_factor_regression(ret_quality)
print(model_full.summary())


                            OLS Regression Results                            
Dep. Variable:                Quality   R-squared:                       0.601
Model:                            OLS   Adj. R-squared:                  0.600
Method:                 Least Squares   F-statistic:                     132.7
Date:                Mon, 08 Jun 2026   Prob (F-statistic):          7.06e-108
Time:                        13:14:20   Log-Likelihood:                 24490.
No. Observations:                4995   AIC:                        -4.897e+04
Df Residuals:                    4990   BIC:                        -4.894e+04
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           4.907e-05    3.1e-05      1.

### Rolling collinearity


In [5]:
vif_ts, cond_ts = rolling_collinearity(
    ret_quality[[c for c in ret_quality.columns if c != "Quality"]],
)

# plot
fig = go.Figure()
for col in vif_ts.columns:
    fig.add_trace(go.Scatter(x=vif_ts.index, y=vif_ts[col], name=col, mode="lines"))
fig.update_layout(
    template="plotly_dark",
    margin=dict(l=40, t=50, b=40, r=40),
    title="Rolling collinearity (VIF)",
)
fig.update_xaxes(title="Date")
fig.show()


### Rolling correlation


In [6]:
window = 256
corr_ts = rolling_correlation(
    ret_quality[[c for c in ret_quality.columns if c != "Quality"]], window=window
)

display(corr_ts.head())

# --- 主要ペアの抽出 ---
peak_abs = corr_ts.abs().max().sort_values(ascending=False)  # 最大絶対相関
sign_flip = corr_ts.apply(lambda s: (s.min() < 0) and (s.max() > 0))  # 符号反転の有無
span = (corr_ts.max() - corr_ts.min()).sort_values(ascending=False)  # 変動レンジ

# print("=== 最大絶対相関 Top5 ===")
# display(peak_abs.head(5).round(2))
# print("\n=== 符号反転したペア ===")
# display(sign_flip[sign_flip].index.tolist())
# print("\n=== 変動レンジ Top5 ===")
# display(span.head(5).round(2))

# --- プロット: 変動の大きい主要ペアのみ ---
key_pairs = span.head(6).index.tolist()
fig = go.Figure()
for col in key_pairs:
    fig.add_trace(go.Scatter(x=corr_ts.index, y=corr_ts[col], name=col, mode="lines"))
fig.update_layout(
    title=f"Factor LS Return: Rolling correlation (window={window} days)",
    template="plotly_dark",
    margin=dict(l=40, t=40, b=40, r=40),
)
fig.show()


,Growth × Low Volatility,Growth × Momentum,Growth × Size,Growth × Value,Low Volatility × Momentum,Low Volatility × Size,Low Volatility × Value,Momentum × Size,Momentum × Value,Size × Value
2007-12-27,0.655276,0.346591,-0.234150,0.430878,0.429050,-0.247075,0.368268,0.275665,0.368273,0.172457
2007-12-28,0.655374,0.346849,-0.235144,0.432566,0.428965,-0.246909,0.368315,0.275325,0.369265,0.174811
2007-12-31,0.655044,0.349324,-0.234049,0.435210,0.426510,-0.253482,0.364832,0.272810,0.368111,0.170918
2008-01-02,0.651673,0.345773,-0.236206,0.435085,0.419807,-0.254481,0.356704,0.274243,0.356848,0.168850
2008-01-03,0.655003,0.348690,-0.234928,0.434285,0.411646,-0.262243,0.347974,0.277727,0.359235,0.176338


### ファクターLSのボラティリティ


In [11]:
window = 256
vol_factors = (
    ret_quality.rolling(window=window, min_periods=window).std() * np.sqrt(252)
).dropna(how="all")

vol_factors.rename(
    columns={
        key: value
        for key, value in zip(
            vol_factors.columns, [c + "_vol" for c in vol_factors.columns]
        )
    },
    inplace=True,
)

vol_factors = pd.merge(
    vol_factors,
    df_cum[
        [
            "FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return",
            "FTW MXWD Index Sector Neutralized Value Long-Short (Low-High) Total Return",
        ]
    ],
    left_index=True,
    right_index=True,
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
for col in [
    c
    for c in vol_factors.columns
    if (c.endswith("_vol")) & any(f in c for f in ["Quality", "Value"])
]:
    fig.add_trace(
        go.Scatter(x=vol_factors.index, y=vol_factors[col], name=col, mode="lines"),
        secondary_y=False,
    )

for col in [c for c in vol_factors.columns if not c.endswith("_vol")]:
    if "Quality" in col:
        color = "white"
    elif "Value" in col:
        color = "gray"
    fig.add_trace(
        go.Scatter(
            x=vol_factors.index,
            y=vol_factors[col],
            name=col,
            mode="lines",
            line=dict(color=color, width=2.5),
        ),
        secondary_y=True,
    )


fig.update_layout(
    title=f"Factor Return and Rolling Volatility (window: {window} days)",
    xaxis_title="Date",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=50, r=50, t=50, b=40),
)
fig.update_yaxes(title_text="Annualized Volatility", secondary_y=False)
fig.update_yaxes(title_text="Factor LS Total Return", range=[-85, 45], secondary_y=True)


fig.show()


#### ローリングでOLS


In [8]:
window = 256
df_rolling_res = run_rolling_factor_regression(
    df_daily_return=ret_quality, window=window, maxlags=21
)


fig = make_subplots(specs=[[{"secondary_y": True}]])

# ファクターベータと決定係数の時系列推移
# coef_cols = [c for c in df_rolling_res.columns if "coef_" in c and "const" not in c]
# coef_cols = ["coef_Value", "coef_Growth"]
# for col in coef_cols:
#     fig.add_trace(
#         go.Scatter(
#             x=df_rolling_res.index,
#             y=df_rolling_res[col],
#             mode="lines",
#             name=col.replace("coef_", ""),
#             opacity=0.7,
#         ),
#         secondary_y=False,
#     )

fig.add_trace(
    go.Scatter(
        x=df_rolling_res.index,
        y=df_rolling_res["coef_const"] * 252,
        mode="lines",
        name="Annualized Alpha",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=df_rolling_res.index,
        y=df_rolling_res["R_squared"],
        mode="lines",
        name="R-squared(RHS)",
        line=dict(color="white", width=2.5, dash="dash"),
    ),
    secondary_y=True,
)


fig.update_layout(
    title=f"Rolling Factor Betas(LHS) and R-squared(RHS) (window: {window} days)",
    xaxis_title="Date",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=50, r=50, t=50, b=40),
)

fig.update_yaxes(title_text="Beta Coefficient", secondary_y=False)
fig.update_yaxes(title_text="R-squared", range=[0, 1], secondary_y=True)

fig.show()


## 複数のウィンドウサイズでのalphaをcheck


In [10]:
window = 256
df_rolling_res = run_rolling_factor_regression(
    df_daily_return=ret_quality, window=window, maxlags=21
)
df_rolling_res_with_return = pd.merge(
    df_rolling_res,
    df_cum[
        ["FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return"]
    ],
    left_index=True,
    right_index=True,
)


fig = make_subplots(specs=[[{"secondary_y": True}]])

# ファクターベータと決定係数の時系列推移
# coef_cols = [c for c in df_rolling_res.columns if "coef_" in c and "const" not in c]
# coef_cols = ["coef_Value", "coef_Growth"]
# for col in coef_cols:
#     fig.add_trace(
#         go.Scatter(
#             x=df_rolling_res.index,
#             y=df_rolling_res[col],
#             mode="lines",
#             name=col.replace("coef_", ""),
#             opacity=0.7,
#         ),
#         secondary_y=False,
#     )

fig.add_trace(
    go.Scatter(
        x=df_rolling_res_with_return.index,
        y=df_rolling_res_with_return["coef_const"] * 252,
        mode="lines",
        name="Annualized Alpha",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=df_rolling_res_with_return.index,
        y=df_rolling_res_with_return[
            "FTW MXWD Index Sector Neutralized Quality Long-Short (High-Low) Total Return"
        ],
        mode="lines",
        name="Quality LS Total Return(RHS)",
        line=dict(color="white"),
    ),
    secondary_y=True,
)


fig.update_layout(
    title=f"Annualized Alpha(LHS) and Quality LS Total Return(RHS) (window: {window} days)",
    xaxis_title="Date",
    template="plotly_dark",
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    margin=dict(l=50, r=50, t=50, b=40),
)

fig.update_yaxes(title_text="Annualized Alpha", secondary_y=False)
fig.update_yaxes(title_text="Factor Return", secondary_y=True)

fig.show()
